# Fallstudie 1 – Unsupervised Learning und Feature Engineering

## Psychische Gesundheit in technologiebezogenen Berufen (DLBDSMLUSL01_D)

**Verfasser:** Andrei Wilke
**Matrikelnummer:** IU14128673
**Kurs:** DLBDSMLUSL01_D – Maschinelles Lernen: Unsupervised Learning und Feature Engineering
**Datum:** 15.04.2026

---

Dieses Notebook bildet die Fallstudie reproduzierbar von Anfang bis Ende ab:

- Datenverständnis und EDA
- Feature Engineering und begründete Merkmalsauswahl
- Clustering-Diagnostik über k = 2 bis k = 8
- Methodenvergleich: k-Means (Baseline) vs. PAM/FasterPAM auf Gower-Distanz (Hauptverfahren) vs. Hierarchisches Clustering auf Gower-Distanz (Zweitmeinung)
- **Finale Modellwahl: PAM / FasterPAM auf Gower-Distanz mit k = 2**
- PCA-Visualisierung
- Profilierungsvariablen für die inhaltliche Interpretation
- Ableitung von HR-Maßnahmen

**Ausgaben:** Das Notebook schreibt zentrale Ergebnisse nach:

- `reports/tables/` (CSV-Tabellen)
- `reports/figures/` (PNG-Grafiken)
- `artifacts/checkpoints/` (DataFrame- und Objekt-Checkpoints)

Das Notebook läuft von oben nach unten ohne manuelle Eingriffe. Pfade werden automatisch relativ zum Notebook-Speicherort aufgelöst; die Ausgabeverzeichnisse werden bei Bedarf angelegt.


## Setup und Bibliotheken

In [1]:
import matplotlib
matplotlib.use('Agg')  # headless-Modus fuer reproduzierbare Ausfuehrung ohne GUI

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Mixed-type-Clustering: Gower-Distanz + PAM (FasterPAM) + Hierarchisches Clustering
import gower
from kmedoids import fasterpam
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
sns.set_context('talk')
sns.set_style('whitegrid')
RNG = 42
print('Setup ok.')

Setup ok.


## Pfadauflösung

Die Pfadauflösung sucht ausgehend vom Notebook-Verzeichnis nach oben, bis sie die erwarteten Ordner findet.  
Dadurch funktioniert das Notebook unabhängig davon, wo es auf dem Rechner liegt.

In [2]:
def find_input_root(start: Path) -> Path:
    """Sucht den Projektordner, der data/raw/mental_health_tech_2016.csv enthält."""
    for candidate in [start] + list(start.parents):
        if (candidate / "data" / "raw" / "mental_health_tech_2016.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Eingabepfad mit data/raw/mental_health_tech_2016.csv nicht gefunden. "
        f"Gesucht ab: {start}"
    )


def find_output_root(start: Path) -> Path:
    """Sucht den Projektordner mit data/ und reports/."""
    for candidate in [start] + list(start.parents):
        if (
            (candidate / "data").exists()
            and (candidate / "reports").exists()
        ):
            return candidate
    return start


NOTEBOOK_DIR = Path.cwd()
INPUT_ROOT = find_input_root(NOTEBOOK_DIR)
OUTPUT_ROOT = find_output_root(NOTEBOOK_DIR)

DATA_FILE       = INPUT_ROOT  / "data" / "raw" / "mental_health_tech_2016.csv"
REPORTS_TABLES  = OUTPUT_ROOT / "reports" / "tables"
REPORTS_FIGURES = OUTPUT_ROOT / "reports" / "figures"
CHECKPOINT_DIR  = OUTPUT_ROOT / "artifacts" / "checkpoints"

# Ausgabeverzeichnisse werden bei der Ausfuehrung lokal erzeugt.
for d in [REPORTS_TABLES, REPORTS_FIGURES, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Anzeige in relativer Form, damit keine Host- oder Session-Pfade im Output stehen.
print("Projektordner :", INPUT_ROOT.name)
print("Datendatei    :", DATA_FILE.relative_to(INPUT_ROOT))
print("Datei vorhanden:", DATA_FILE.exists())


Projektordner : DLBDSMLUSL01_IU14128673_Andrei Wilke_Fallstudie1
Datendatei    : data/raw/mental_health_tech_2016.csv
Datei vorhanden: True


## Datensatz laden

In [3]:
df_raw = pd.read_csv(DATA_FILE)
df = df_raw.copy()

print("Datensatz geladen.")
print("Shape:", df.shape)
df.head(3)

Datensatz geladen.
Shape: (1433, 63)


,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",...,"If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?","If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",What is your age?,What is your gender?,What country do you live in?,What US state or territory do you live in?,What country do you work in?,What US state or territory do you work in?,Which of the following best describes your work position?,Do you work remotely?
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never
2,0,6-25,1.0,NaN,No,NaN,No,No,I don't know,Neither easy nor difficult,...,Not applicable to me,Not applicable to me,38,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Always


## 1) EDA – Grundstruktur und Missing-Sichtung

Die EDA dient nicht nur der Beschreibung, sondern vor allem der methodischen Entscheidungsvorbereitung.  
Zentraler Befund: Ein erheblicher Teil der Missing Values folgt der Fragebogenlogik (strukturelle Folgefragen,  
Selbstständigen-Blöcke, Arbeitgeber-Blöcke) und ist damit inhaltlich interpretierbar, nicht zufällig.

In [4]:
sighting = pd.DataFrame({
    "variable":     df.columns,
    "dtype":        df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct":  (df.isna().mean() * 100).round(2).values,
    "n_unique":     df.nunique(dropna=True).values,
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False]).reset_index(drop=True)

sighting.to_csv(REPORTS_TABLES / "01_sichtungsprotokoll.csv", index=False)
print(f"Spalten mit Missing > 0 %: {int((sighting['missing_pct'] > 0).sum())}")
print(f"Spalten ohne Missing    : {int((sighting['missing_pct'] == 0).sum())}")
print("Gespeichert:", (REPORTS_TABLES / '01_sichtungsprotokoll.csv').relative_to(OUTPUT_ROOT))
sighting.head(15)

Spalten mit Missing > 0 %: 44
Spalten ohne Missing    : 19
Gespeichert: reports/tables/01_sichtungsprotokoll.csv


,variable,dtype,missing_count,missing_pct,n_unique
0,If you have revealed a mental health issue to ...,object,1289,89.95,3
1,"If yes, what percentage of your work time (tim...",object,1229,85.76,4
2,Is your primary role within your company relat...,float64,1170,81.65,2
3,If you have been diagnosed or treated for a me...,object,1146,79.97,5
4,If you have been diagnosed or treated for a me...,object,1146,79.97,5
5,If you have revealed a mental health issue to ...,object,1146,79.97,4
6,Do you believe your productivity is ever affec...,object,1146,79.97,4
7,Do you know local or online resources to seek ...,object,1146,79.97,3
8,Do you have medical coverage (private insuranc...,float64,1146,79.97,2
9,"If maybe, what condition(s) do you believe you...",object,1111,77.53,99


In [5]:
# Überblick über wichtige Strukturvariablen
key_cols = [
    "What is your age?",
    "What is your gender?",
    "Which of the following best describes your work position?",
    "Do you have previous employers?",
    "Are you self-employed?",
]
for col in key_cols:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False).head(8).to_string())


What is your age?
What is your age?
30    94
31    82
29    79
28    74
35    74
32    72
34    69
33    69

What is your gender?
What is your gender?
Male      610
male      249
Female    153
female     95
M          86
m          79
F          38
f          23

Which of the following best describes your work position?
Which of the following best describes your work position?
Back-end Developer                        263
Front-end Developer                       125
Other                                     112
Supervisor/Team Lead                       68
Back-end Developer|Front-end Developer     61
DevOps/SysAdmin                            54
One-person shop                            50
Executive Leadership                       46

Do you have previous employers?
Do you have previous employers?
1    1264
0     169

Are you self-employed?
Are you self-employed?
0    1146
1     287


## 2) Entscheidungstabelle für den Clusterkern

Variablen werden in vier Klassen eingeteilt:
- **A_core_direct**: direkt clusterrelevant (nach Transformation)
- **B_core_after_transform**: clusterrelevant nach Transformation (Alter, Geschlecht, Rolle)
- **C_profile_only**: nur für nachgelagerte Profilierung und Interpretation
- **D_exclude_structural**: strukturell auszuschließen (Fragebogenlogik)

In [6]:
def canonical(text: str) -> str:
    """Normalisiert Spaltennamen für robuste Suche."""
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()

column_lookup = {canonical(c): c for c in df.columns}

def resolve_column(name: str) -> str:
    """Gibt den echten Spaltennamen zurück – robust gegen Sonderzeichen und Leerzeichen."""
    key = canonical(name)
    if key in column_lookup:
        return column_lookup[key]
    contains_matches = [orig for k, orig in column_lookup.items() if key in k or k in key]
    if len(contains_matches) == 1:
        return contains_matches[0]
    raise KeyError(f"Spalte nicht eindeutig auffindbar: {name!r}")


A_core_direct_raw = [
    "Would you be willing to bring up a physical health issue with a potential employer in an interview?",
    "Would you bring up a mental health issue with a potential employer in an interview?",
    "Do you feel that being identified as a person with a mental health issue would hurt your career?",
    "Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?",
    "How willing would you be to share with friends and family that you have a mental illness?",
    "Do you have a family history of mental illness?",
    "Have you had a mental health disorder in the past?",
    "Do you currently have a mental health disorder?",
    "Have you been diagnosed with a mental health condition by a medical professional?",
    "Have you ever sought treatment for a mental health issue from a mental health professional?",
    "If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?",
    "If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?",
    "Do you work remotely?",
]
B_core_after_transform_raw = [
    "What is your age?",
    "What is your gender?",
    "Which of the following best describes your work position?",
]
C_profile_only_raw = [
    "Are you self-employed?",
    "Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?",
    "What country do you live in?",
    "What country do you work in?",
]
D_exclude_structural_raw = [
    "Do you have previous employers?",
]

A_core_direct        = [resolve_column(x) for x in A_core_direct_raw]
B_core_after_transform = [resolve_column(x) for x in B_core_after_transform_raw]
C_profile_only       = [resolve_column(x) for x in C_profile_only_raw]
D_exclude_structural = [resolve_column(x) for x in D_exclude_structural_raw]

decision_rows = []
for col in A_core_direct:        decision_rows.append({"variable": col, "decision_class": "A_core_direct"})
for col in B_core_after_transform: decision_rows.append({"variable": col, "decision_class": "B_core_after_transform"})
for col in C_profile_only:       decision_rows.append({"variable": col, "decision_class": "C_profile_only"})
for col in D_exclude_structural:  decision_rows.append({"variable": col, "decision_class": "D_exclude_structural"})

decision_df = pd.DataFrame(decision_rows)
assert decision_df["variable"].is_unique, "Doppelte Variable in Entscheidungstabelle!"

decision_df.to_csv(REPORTS_TABLES / "02_decision_df.csv", index=False)
print("Entscheidungsklassen:")
print(decision_df["decision_class"].value_counts().to_string())
print("\nGespeichert:", (REPORTS_TABLES / "02_decision_df.csv").relative_to(OUTPUT_ROOT))
decision_df

Entscheidungsklassen:
decision_class
A_core_direct             13
C_profile_only             4
B_core_after_transform     3
D_exclude_structural       1

Gespeichert: reports/tables/02_decision_df.csv


,variable,decision_class
0,Would you be willing to bring up a physical he...,A_core_direct
1,Would you bring up a mental health issue with ...,A_core_direct
2,Do you feel that being identified as a person ...,A_core_direct
3,Do you think that team members/co-workers woul...,A_core_direct
4,How willing would you be to share with friends...,A_core_direct
5,Do you have a family history of mental illness?,A_core_direct
6,Have you had a mental health disorder in the p...,A_core_direct
7,Do you currently have a mental health disorder?,A_core_direct
8,Have you been diagnosed with a mental health c...,A_core_direct
9,Have you ever sought treatment for a mental he...,A_core_direct


## 3) Feature Engineering

### 3.1 Alter, Geschlecht, Arbeitsrolle

In [7]:
# --- Alter ---
age_var = resolve_column("What is your age?")
df["age_clean"]   = df[age_var].where(df[age_var].between(18, 80), pd.NA).astype("float")
age_median        = df["age_clean"].median()
df["age_imputed"] = df["age_clean"].fillna(age_median)
df["age_scaled"]  = StandardScaler().fit_transform(df[["age_imputed"]]).ravel()

print(f"Alter – unplausible Werte (<18 oder >80): {int(df['age_clean'].isna().sum())}")
print(f"Imputationsmedian: {float(age_median):.1f}")
df[[age_var, "age_clean", "age_imputed", "age_scaled"]].head()

Alter – unplausible Werte (<18 oder >80): 5
Imputationsmedian: 33.0


,What is your age?,age_clean,age_imputed,age_scaled
0,39,39.0,39.0,0.609407
1,29,29.0,29.0,-0.629815
2,38,38.0,38.0,0.485484
3,43,43.0,43.0,1.105095
4,43,43.0,43.0,1.105095


In [8]:
# --- Geschlecht ---
gender_var = resolve_column("What is your gender?")

def normalize_gender(x):
    """Normalisiert Freitexteingaben: lowercase, Sonderzeichen zu Whitespace, trim."""
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9\s\-/]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

# Praezise Wort-Boundary-Regex vermeidet den Substring-Fehler (female enthaelt male).
# Reihenfolge: zuerst nonbinary/diverse, dann entweder female XOR male, sonst unknown_other.
_NB_PATTERNS = [
    r"\bnonbinary\b", r"\bnon binary\b", r"\bnon-binary\b",
    r"\bgenderqueer\b", r"\bagender\b", r"\bgender ?fluid\b", r"\bfluid\b",
    r"\benby\b", r"\bbigender\b", r"\bandrogynous\b",
    r"\btrans\b", r"\bmtf\b", r"\bftm\b", r"\btwo[- ]spirit\b",
    r"\bgender nonconforming\b", r"\bqueer\b",
]
_FEMALE_PATTERN = re.compile(r"\bfemale\b|\bwoman\b|\bwomen\b|\bfem\b|\bfm\b|^f$")
_MALE_PATTERN   = re.compile(r"\bmale\b|\bman\b|\bmen\b|^m$|^mail$|^malr$|^dude$")
_NB_RE = [re.compile(p) for p in _NB_PATTERNS]

def map_gender_group(x):
    """Robuste Gender-Klassifikation mit Wort-Boundaries.

    Warum Regex statt 'in'?  Der Teilstring 'male' ist in 'female' enthalten;
    eine naive Substring-Logik klassifiziert 'female' daher faelschlich als
    'male' oder als 'unknown_other'.  Wort-Boundaries (\\b) verhindern das.
    Findet sich sowohl ein eindeutiger weiblicher als auch ein eindeutiger
    maennlicher Marker (z. B. 'male 9:1 female, roughly'), wird die Angabe
    konservativ als 'unknown_other' gefuehrt.
    """
    x = normalize_gender(x)
    if not x:
        return "unknown_other"
    if any(r.search(x) for r in _NB_RE):
        return "nonbinary_diverse"
    has_female = bool(_FEMALE_PATTERN.search(x))
    has_male = bool(_MALE_PATTERN.search(x))
    if has_female and not has_male:
        return "female"
    if has_male and not has_female:
        return "male"
    return "unknown_other"

df["gender_grouped"] = df[gender_var].apply(map_gender_group)
gender_dummies = pd.get_dummies(df["gender_grouped"], prefix="gender", dtype=int)
for col in ["gender_female", "gender_male", "gender_nonbinary_diverse", "gender_unknown_other"]:
    if col not in gender_dummies.columns:
        gender_dummies[col] = 0
gender_dummies = gender_dummies[
    ["gender_female", "gender_male", "gender_nonbinary_diverse", "gender_unknown_other"]
]
df = pd.concat([df, gender_dummies], axis=1)

print("Geschlechtsgruppen (nach robuster Normalisierung):")
print(df["gender_grouped"].value_counts().to_string())


Geschlechtsgruppen (nach robuster Normalisierung):
gender_grouped
male                 1056
female                341
nonbinary_diverse      21
unknown_other          15


In [9]:
# --- Arbeitsrolle (Multi-Hot) ---
work_var = resolve_column("Which of the following best describes your work position?")

def tokenize_work(text):
    if pd.isna(text): return []
    return [re.sub(r"\s+", " ", t).strip()
            for t in re.split(r"[\|,;/+&]", str(text).lower().strip())
            if str(t).strip()]

def has_keyword(tokens, keywords):
    return int(any(any(k in tok for k in keywords) for tok in tokens))

work_tokens = df[work_var].apply(tokenize_work)

df["role_backend"]        = work_tokens.apply(lambda t: has_keyword(t, ["back-end developer", "backend", "back end developer"]))
df["role_frontend"]       = work_tokens.apply(lambda t: has_keyword(t, ["front-end developer", "frontend", "front end developer"]))
df["role_devops"]         = work_tokens.apply(lambda t: has_keyword(t, ["devops", "sysadmin", "sys admin", "system administrator"]))
df["role_leadership"]     = work_tokens.apply(lambda t: has_keyword(t, ["supervisor", "team lead", "executive leadership", "leadership"]))
df["role_enabling"]       = work_tokens.apply(lambda t: has_keyword(t, ["support", "designer", "dev evangelist", "advocate"]))
df["role_one_person_shop"]= work_tokens.apply(lambda t: has_keyword(t, ["one-person shop", "one person shop"]))
df["role_people_business"]= work_tokens.apply(lambda t: has_keyword(t, ["sales", "hr", "human resources", "people operations"]))
df["role_other"]          = work_tokens.apply(lambda t: has_keyword(t, ["other"]))

role_cols = ["role_backend", "role_frontend", "role_devops", "role_leadership",
             "role_enabling", "role_one_person_shop", "role_people_business", "role_other"]
df["work_role_count"] = df[role_cols].sum(axis=1)
df["work_multi_role"] = (df["work_role_count"] > 1).astype(int)

print("Rollen-Features (Summen):")
print(df[role_cols + ["work_multi_role"]].sum().to_string())

Rollen-Features (Summen):
role_backend            737
role_frontend           502
role_devops             282
role_leadership         340
role_enabling           343
role_one_person_shop    161
role_people_business     40
role_other              187
work_multi_role         608


### 3.2 Kodierung der 13 Kernvariablen (A_core_direct)

In [10]:
# Spaltenzuordnung
var_phys_interview   = resolve_column("Would you be willing to bring up a physical health issue with a potential employer in an interview?")
var_mental_interview = resolve_column("Would you bring up a mental health issue with a potential employer in an interview?")
var_career_hurt      = resolve_column("Do you feel that being identified as a person with a mental health issue would hurt your career?")
var_coworker_neg     = resolve_column("Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?")
var_share_family     = resolve_column("How willing would you be to share with friends and family that you have a mental illness?")
var_family_history   = resolve_column("Do you have a family history of mental illness?")
var_mh_past          = resolve_column("Have you had a mental health disorder in the past?")
var_mh_current       = resolve_column("Do you currently have a mental health disorder?")
var_mh_diagnosed     = resolve_column("Have you been diagnosed with a mental health condition by a medical professional?")
var_mh_treatment     = resolve_column("Have you ever sought treatment for a mental health issue from a mental health professional?")
var_interfere_treated   = resolve_column("If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?")
var_interfere_untreated = resolve_column("If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?")
var_remote_work      = resolve_column("Do you work remotely?")


def norm(v):
    if pd.isna(v): return ""
    return re.sub(r"\s+", " ", str(v)).strip()

def map_three_level(v):
    return {"No": 0, "Maybe": 1, "Yes": 2}.get(norm(v), np.nan)

def map_stigma(v):
    txt = norm(v).lower()
    if txt.startswith("no"): return 0
    if txt.startswith("maybe"): return 1
    if txt.startswith("yes"): return 2
    return np.nan

def map_share(v):
    txt = norm(v).lower()
    if "not open at all" in txt: return 0
    if "somewhat not open" in txt: return 1
    if txt == "neutral": return 2
    if "somewhat open" in txt: return 3
    if "very open" in txt: return 4
    return np.nan

def map_family(v):
    return {"No": 0, "I don't know": 1, "Yes": 2}.get(norm(v), np.nan)

def map_binary_yes(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return 1 if v == 1 else (0 if v == 0 else np.nan)
    txt = norm(v).lower()
    if txt in {"1", "1.0", "true", "yes"} or txt.startswith("yes"): return 1
    if txt in {"0", "0.0", "false", "no"} or txt.startswith("no"): return 0
    return np.nan

def split_interfere(v):
    txt = norm(v).lower()
    na_flag = int("not applicable" in txt)
    if "never" in txt:     ord_val = 0
    elif "rarely" in txt:  ord_val = 1
    elif "sometimes" in txt: ord_val = 2
    elif "often" in txt:   ord_val = 3
    elif na_flag == 1:     ord_val = 0
    else:                  ord_val = np.nan
    return ord_val, na_flag


# Anwendung der Kodierfunktionen
for source, target in [
    (var_phys_interview,   "phys_interview_open_ord"),
    (var_mental_interview, "mental_interview_open_ord"),
    (var_mh_past,          "mh_past_ord"),
    (var_mh_current,       "mh_current_ord"),
]:
    df[target] = df[source].apply(map_three_level)

df["career_hurt_ord"]      = df[var_career_hurt].apply(map_stigma)
df["coworker_negative_ord"]= df[var_coworker_neg].apply(map_stigma)
df["share_family_open_ord"]= df[var_share_family].apply(map_share)
df["share_family_na"]      = df["share_family_open_ord"].isna().astype(int)
df["family_history_ord"]   = df[var_family_history].apply(map_family)
df["mh_diagnosed_bin"]     = df[var_mh_diagnosed].apply(map_binary_yes)
df["mh_treatment_bin"]     = df[var_mh_treatment].apply(map_binary_yes)

treated_split   = df[var_interfere_treated].apply(split_interfere)
untreated_split = df[var_interfere_untreated].apply(split_interfere)
df["interfere_treated_ord"]   = treated_split.apply(lambda x: x[0])
df["interfere_treated_na"]    = treated_split.apply(lambda x: x[1])
df["interfere_untreated_ord"] = untreated_split.apply(lambda x: x[0])
df["interfere_untreated_na"]  = untreated_split.apply(lambda x: x[1])
df["remote_work_ord"]         = df[var_remote_work].map({"Never": 0, "Sometimes": 1, "Always": 2})

encoded_features = [
    "phys_interview_open_ord", "mental_interview_open_ord",
    "career_hurt_ord", "coworker_negative_ord",
    "share_family_open_ord", "share_family_na",
    "family_history_ord", "mh_past_ord", "mh_current_ord",
    "mh_diagnosed_bin", "mh_treatment_bin",
    "interfere_treated_ord", "interfere_treated_na",
    "interfere_untreated_ord", "interfere_untreated_na",
    "remote_work_ord",
]
for col in encoded_features:
    if df[col].isna().any():
        fill_val = df[col].mode(dropna=True).iloc[0] if not df[col].dropna().empty else 0
        df[col] = df[col].fillna(fill_val)

print("Kodierte Kernfeatures – verbleibende Missing-Werte:")
missing_check = df[encoded_features].isna().sum()
print(missing_check[missing_check > 0].to_string() if missing_check.any() else "Keine Missing-Werte.")

Kodierte Kernfeatures – verbleibende Missing-Werte:
Keine Missing-Werte.


### 3.3 Finale Kernmatrizen (V5)

In [11]:
cluster_core_v5_primary = [
    "age_scaled", "gender_female",
    "role_backend", "role_frontend", "role_devops", "role_leadership",
    "role_enabling", "role_one_person_shop", "work_multi_role",
    "phys_interview_open_ord", "mental_interview_open_ord",
    "career_hurt_ord", "coworker_negative_ord",
    "share_family_open_ord", "share_family_na",
    "family_history_ord", "mh_past_ord", "mh_current_ord",
    "mh_diagnosed_bin", "mh_treatment_bin",
    "interfere_treated_ord", "interfere_treated_na",
    "interfere_untreated_ord", "interfere_untreated_na",
    "remote_work_ord",
]
cluster_core_v5_sensitivity = cluster_core_v5_primary + [
    "gender_male", "gender_nonbinary_diverse", "role_people_business",
]

core_df_primary     = df[cluster_core_v5_primary].copy()
core_df_sensitivity = df[cluster_core_v5_sensitivity].copy()

for name, frame in [("primary", core_df_primary), ("sensitivity", core_df_sensitivity)]:
    if frame.isna().any().any():
        frame.fillna(frame.median(numeric_only=True), inplace=True)
    print(f"{name}: shape={frame.shape}, missing={int(frame.isna().sum().sum())}")

X_v5_primary = core_df_primary.copy()
scaler_v5    = StandardScaler()
X_v5_scaled  = scaler_v5.fit_transform(X_v5_primary)
print(f"Skalierte Eingabematrix: {X_v5_scaled.shape}")

primary: shape=(1433, 25), missing=0
sensitivity: shape=(1433, 28), missing=0
Skalierte Eingabematrix: (1433, 25)


## 4) Clusterverfahren: methodischer Vergleich (k=2 bis k=8)

Der Merkmalsraum besteht aus 25 gemischten Features: eine stetige Groesse (standardisiertes Alter),
13 ordinalskalierte Variablen und 11 binaere Indikatoren. Auf einem rein euklidischen Raum bewertet
k-Means Distanzen zwischen ordinalen Stufen wie metrische Differenzen und wertet binaere Flags
durch die Standardisierung kuenstlich auf. Fuer gemischte Variablen ist die **Gower-Distanz** das
etablierte Mass, das numerische, ordinale und binaere Merkmale jeweils geeignet normalisiert und
gleichgewichtet aggregiert. Passend dazu wird **PAM / k-Medoids** (FasterPAM nach Schubert und
Rousseeuw, 2021) eingesetzt, das mit beliebigen Distanzen arbeitet und interpretierbare Medoids
– also reale Befragte – als Clusterzentren zurueckgibt.

Verglichen werden drei Verfahren auf identischer Datenmatrix:

1. **k-Means (euklidisch, StandardScaler)** – bisherige Baseline
2. **PAM / FasterPAM auf Gower-Distanz** – Verfahren fuer gemischte Merkmalstypen
3. **Hierarchisches Clustering (average-link) auf Gower-Distanz** – robuste Zweitmeinung

Vier Gueteindices werden einheitlich berichtet: Inertia (soweit definiert) fuer den Ellbogen,
Silhouette, Calinski-Harabasz und Davies-Bouldin. Die Silhouette fuer PAM und HAC wird
distanzkonsistent auf der Gower-Matrix berechnet, zusaetzlich wird zur Vergleichbarkeit auch
die euklidische Silhouette auf demselben Merkmalsraum ausgewiesen.

In [12]:
# Gower-Distanzmatrix: einmal berechnen und wiederverwenden.
# cat_features markiert binaere Features (dummies + NA-Flags) als kategorial;
# stetige und ordinale Merkmale bleiben numerisch und werden von Gower
# auf [0,1] range-normalisiert.
_categorical_flags = {
    'gender_female', 'role_backend', 'role_frontend', 'role_devops',
    'role_leadership', 'role_enabling', 'role_one_person_shop', 'work_multi_role',
    'share_family_na', 'mh_diagnosed_bin', 'mh_treatment_bin',
    'interfere_treated_na', 'interfere_untreated_na',
}
cat_mask = np.array([c in _categorical_flags for c in cluster_core_v5_primary])
D_gower = gower.gower_matrix(X_v5_primary.values, cat_features=cat_mask)
print(f'Gower-Distanzmatrix: {D_gower.shape}, '
      f'Wertebereich [{D_gower.min():.3f}, {D_gower.max():.3f}]')

# Gueteindices fuer alle drei Verfahren ueber k=2..8
method_rows = []

# Verfahren 1: k-Means (Baseline, euklidisch)
for k in range(2, 9):
    m = KMeans(n_clusters=k, random_state=RNG, n_init=50).fit(X_v5_scaled)
    lab = m.labels_
    method_rows.append({
        'method': 'kmeans_euclid', 'k': k,
        'inertia': float(m.inertia_),
        'silhouette_native': silhouette_score(X_v5_scaled, lab),
        'silhouette_eucl':   silhouette_score(X_v5_scaled, lab),
        'calinski_harabasz': calinski_harabasz_score(X_v5_scaled, lab),
        'davies_bouldin':    davies_bouldin_score(X_v5_scaled, lab),
    })

# Verfahren 2: PAM / FasterPAM auf Gower-Distanz
for k in range(2, 9):
    res = fasterpam(D_gower, k, random_state=RNG)
    lab = np.asarray(res.labels)
    method_rows.append({
        'method': 'pam_gower', 'k': k,
        'inertia': float(res.loss),
        'silhouette_native': silhouette_score(D_gower, lab, metric='precomputed'),
        'silhouette_eucl':   silhouette_score(X_v5_scaled, lab),
        'calinski_harabasz': calinski_harabasz_score(X_v5_scaled, lab),
        'davies_bouldin':    davies_bouldin_score(X_v5_scaled, lab),
    })

# Verfahren 3: Hierarchisches Clustering (average-link) auf Gower-Distanz
Z_hac = linkage(squareform(D_gower, checks=False), method='average')
for k in range(2, 9):
    lab = fcluster(Z_hac, t=k, criterion='maxclust') - 1
    if len(set(lab)) < 2:
        continue
    method_rows.append({
        'method': 'hac_gower_average', 'k': k,
        'inertia': np.nan,
        'silhouette_native': silhouette_score(D_gower, lab, metric='precomputed'),
        'silhouette_eucl':   silhouette_score(X_v5_scaled, lab),
        'calinski_harabasz': calinski_harabasz_score(X_v5_scaled, lab),
        'davies_bouldin':    davies_bouldin_score(X_v5_scaled, lab),
    })

method_eval_df = pd.DataFrame(method_rows)
method_eval_df.to_csv(REPORTS_TABLES / '03_method_eval.csv', index=False)
print("Gespeichert:", (REPORTS_TABLES / "03_method_eval.csv").relative_to(OUTPUT_ROOT))
method_eval_df.round(3)

Gower-Distanzmatrix: (1433, 1433), Wertebereich [0.000, 0.892]


Gespeichert: reports/tables/03_method_eval.csv


,method,k,inertia,silhouette_native,silhouette_eucl,calinski_harabasz,davies_bouldin
0,kmeans_euclid,2,28372.500,0.203,0.203,375.875,1.866
1,kmeans_euclid,3,26694.047,0.132,0.132,244.573,2.645
2,kmeans_euclid,4,25485.903,0.115,0.115,193.238,2.586
3,kmeans_euclid,5,24576.902,0.106,0.106,163.388,2.527
4,kmeans_euclid,6,23755.846,0.109,0.109,144.997,2.445
5,kmeans_euclid,7,22989.267,0.097,0.097,132.698,2.555
6,kmeans_euclid,8,22375.399,0.095,0.095,122.365,2.452
7,pam_gower,2,308.931,0.358,0.195,366.028,1.921
8,pam_gower,3,279.328,0.241,0.123,231.159,2.769
9,pam_gower,4,268.573,0.184,0.090,178.118,3.136


In [13]:
# Visualisierung: fuer jede Kennzahl eine kleine Panel-Zeile ueber k und Methoden.
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

kennzahlen = [
    ('silhouette_native', 'Silhouette (distanzkonsistent)', axes[0, 0], 'high'),
    ('calinski_harabasz','Calinski-Harabasz (eukl. Raum)',  axes[0, 1], 'high'),
    ('davies_bouldin',   'Davies-Bouldin (eukl. Raum)',     axes[1, 0], 'low'),
    ('inertia',          'Inertia / PAM-Loss (k-spezifisch)', axes[1, 1], 'low'),
]
method_labels = {
    'kmeans_euclid':      'k-Means (Baseline)',
    'pam_gower':          'PAM / Gower',
    'hac_gower_average':  'HAC avg / Gower',
}
colors = {'kmeans_euclid':'#888888', 'pam_gower':'#1f77b4', 'hac_gower_average':'#2ca02c'}

for col, title, ax, direction in kennzahlen:
    for meth, label in method_labels.items():
        sub = method_eval_df[method_eval_df['method'] == meth].sort_values('k')
        ax.plot(sub['k'], sub[col], marker='o', linewidth=2, label=label, color=colors[meth])
    ax.set_title(title); ax.set_xlabel('k'); ax.grid(True, alpha=0.3)
    if direction == 'high':
        ax.set_ylabel('hoeher = besser')
    else:
        ax.set_ylabel('niedriger = besser')
    ax.legend(fontsize=10)

plt.tight_layout()
fig_path = REPORTS_FIGURES / 'method_comparison_overview.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight'); plt.close(fig)
print("Gespeichert:", fig_path.relative_to(OUTPUT_ROOT))

# Zusaetzliche Fokusgrafik: Silhouette-Vergleich bei k=2 bis k=4 als Balken
fig, ax = plt.subplots(figsize=(8, 4.5))
focus = method_eval_df[method_eval_df['k'].between(2, 4)].copy()
focus['method_label'] = focus['method'].map(method_labels)
pivot = focus.pivot(index='k', columns='method_label', values='silhouette_native')
pivot.plot(kind='bar', ax=ax, color=[colors['kmeans_euclid'], colors['pam_gower'], colors['hac_gower_average']])
ax.set_title('Silhouette je Verfahren (k=2..4)')
ax.set_ylabel('Silhouette (distanzkonsistent)')
ax.set_xlabel('k'); ax.set_xticklabels(pivot.index, rotation=0)
ax.legend(fontsize=10)
plt.tight_layout()
fig_path2 = REPORTS_FIGURES / 'method_silhouette_k2_k4.png'
plt.savefig(fig_path2, dpi=140, bbox_inches='tight'); plt.close(fig)
print("Gespeichert:", fig_path2.relative_to(OUTPUT_ROOT))

Gespeichert: reports/figures/method_comparison_overview.png
Gespeichert: reports/figures/method_silhouette_k2_k4.png


## 5) Finale Modellentscheidung und Clusterbildung

Die Diagnostik zeigt konsistent, dass k=2 das robusteste Ergebnis liefert (globales
Silhouette-Maximum ueber alle drei Verfahren, konvergente Minima in Davies-Bouldin und
Calinski-Harabasz-Maximum). Beim Vergleich der Verfahren bei k=2 faellt die
Silhouette auf der Gower-Distanz fuer **PAM/Gower** und **HAC/Gower-average** nahezu
identisch aus und liegt rund 70 bis 75 Prozent hoeher als die euklidische Silhouette
des k-Means-Baselinemodells. PAM wird HAC vorgezogen, weil FasterPAM

1. **globale Optima** deterministisch anstrebt, waehrend average-link-HAC ein greedy-Ansatz ist,
2. **Medoids** – also reale Befragte – als Clusterzentren liefert, was die Interpretation im
   Bericht stuetzt, und
3. im Gegensatz zu HAC keine willkuerliche Cut-off-Hoehe benoetigt.

Finales Verfahren: **PAM / FasterPAM auf Gower-Distanz mit k=2**. k-Means bleibt im
Notebook als nachvollziehbarer Baseline-Vergleich erhalten.

In [14]:
# Finales Modell: PAM auf Gower-Distanz mit k=2
pam_final = fasterpam(D_gower, 2, random_state=RNG)
df['cluster_final'] = np.asarray(pam_final.labels)
medoid_indices = [int(i) for i in pam_final.medoids]
print(f'PAM-Loss: {pam_final.loss:.3f}')
print(f'Medoid-Indizes: {medoid_indices}')

# Sensitivitaetsmodell k=3 (nur PAM, fuer Subclusterblick)
pam_k3 = fasterpam(D_gower, 3, random_state=RNG)
df['cluster_final_k3'] = np.asarray(pam_k3.labels)

# Baseline-Label mitfuehren zur Transparenz im Bericht
kmeans_baseline = KMeans(n_clusters=2, random_state=RNG, n_init=50).fit(X_v5_scaled)
df['cluster_kmeans_baseline'] = kmeans_baseline.labels_

# Clustergroessen
cluster_sizes_k2 = df['cluster_final'].value_counts().sort_index().to_frame('count')
cluster_sizes_k3 = df['cluster_final_k3'].value_counts().sort_index().to_frame('count')
cluster_sizes_k2.to_csv(REPORTS_TABLES / '04_cluster_sizes_k2.csv')
cluster_sizes_k3.to_csv(REPORTS_TABLES / '05_cluster_sizes_k3.csv')

# Clusterprofile auf den 25 Kernfeatures (Mittelwerte)
cluster_profile_k2 = df.groupby('cluster_final')[cluster_core_v5_primary].mean().round(3)
cluster_profile_k3 = df.groupby('cluster_final_k3')[cluster_core_v5_primary].mean().round(3)
cluster_profile_k2.to_csv(REPORTS_TABLES / '06_cluster_profile_k2.csv')
cluster_profile_k3.to_csv(REPORTS_TABLES / '07_cluster_profile_k3.csv')

# Medoids als Tabelle (reale Befragte – interpretierbare Clusterzentren)
medoid_df = X_v5_primary.iloc[medoid_indices].copy()
medoid_df.index = [f'cluster_{c}_medoid_row{idx}' for c, idx in enumerate(medoid_indices)]
medoid_df.to_csv(REPORTS_TABLES / '12_medoids_k2.csv')

# Stabilitaetscheck: mehrere Seeds
stab_rows = []
for seed in (7, 42, 123, 777, 2024):
    res_s = fasterpam(D_gower, 2, random_state=seed)
    lab_s = np.asarray(res_s.labels)
    stab_rows.append({
        'seed': seed,
        'silhouette': silhouette_score(D_gower, lab_s, metric='precomputed'),
        'size_cluster_0': int((lab_s == 0).sum()),
        'size_cluster_1': int((lab_s == 1).sum()),
    })
stability_df = pd.DataFrame(stab_rows)
stability_df.to_csv(REPORTS_TABLES / '13_pam_stability_k2.csv', index=False)

print('Clustergroessen k=2:'); print(cluster_sizes_k2.to_string())
print('\nClustergroessen k=3 (Sensitivitaet):'); print(cluster_sizes_k3.to_string())
print('\nStabilitaet ueber Seeds:'); print(stability_df.to_string(index=False))

PAM-Loss: 308.931
Medoid-Indizes: [617, 1024]


Clustergroessen k=2:
               count
cluster_final       
0                843
1                590

Clustergroessen k=3 (Sensitivitaet):
                  count
cluster_final_k3       
0                   529
1                   545
2                   359

Stabilitaet ueber Seeds:
 seed  silhouette  size_cluster_0  size_cluster_1
    7    0.357965             590             843
   42    0.357965             843             590
  123    0.357965             590             843
  777    0.357965             590             843
 2024    0.357965             843             590


In [15]:
# Staerkste Trennmerkmale zwischen Cluster 0 und Cluster 1 (PAM-Loesung)
cluster_diff_final = (
    cluster_profile_k2.loc[0] - cluster_profile_k2.loc[1]
).sort_values(key=lambda s: s.abs(), ascending=False)

cluster_diff_final.to_frame('cluster0_minus_cluster1').to_csv(
    REPORTS_TABLES / '08_cluster_diff_k2.csv'
)
print('Staerkste Trennmerkmale (PAM k=2):')
cluster_diff_final.to_frame('cluster0_minus_cluster1').head(15)

Staerkste Trennmerkmale (PAM k=2):


,cluster0_minus_cluster1
interfere_untreated_ord,2.088
mh_past_ord,1.381
interfere_treated_ord,1.266
mh_current_ord,1.262
family_history_ord,0.853
interfere_treated_na,-0.826
mh_diagnosed_bin,0.795
interfere_untreated_na,-0.759
mh_treatment_bin,0.756
mental_interview_open_ord,-0.209


## 6) PCA-Visualisierung

In [16]:
pca   = PCA(n_components=2, random_state=RNG)
X_pca = pca.fit_transform(X_v5_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'], index=df.index)
pca_df['cluster_final']           = df['cluster_final'].astype(str)
pca_df['cluster_final_k3']        = df['cluster_final_k3'].astype(str)
pca_df['cluster_kmeans_baseline'] = df['cluster_kmeans_baseline'].astype(str)

explained = pd.DataFrame({
    'component':                ['PC1', 'PC2'],
    'explained_variance_ratio': pca.explained_variance_ratio_,
})
explained.to_csv(REPORTS_TABLES / '09_pca_explained_variance.csv', index=False)
print('PC1/PC2 erklaerte Varianz:', pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, col, title in [
    (axes[0], 'cluster_final',           'PAM/Gower k=2 (final)'),
    (axes[1], 'cluster_final_k3',        'PAM/Gower k=3 (Sensitivitaet)'),
    (axes[2], 'cluster_kmeans_baseline', 'k-Means k=2 (Baseline)'),
]:
    for c in sorted(pca_df[col].unique()):
        sub = pca_df[pca_df[col] == c]
        ax.scatter(sub['PC1'], sub['PC2'], s=18, alpha=0.6, label=f'Cluster {c}')
    ax.set_title(title)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f} %)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f} %)')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Medoids markieren
for idx in medoid_indices:
    axes[0].scatter(X_pca[idx, 0], X_pca[idx, 1], s=260, marker='*',
                    edgecolor='black', facecolor='yellow', linewidth=1.5, zorder=5,
                    label='Medoid' if idx == medoid_indices[0] else None)
axes[0].legend()

plt.tight_layout()
fig_path = REPORTS_FIGURES / 'pca_cluster_final.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight'); plt.close(fig)
print("Gespeichert:", fig_path.relative_to(OUTPUT_ROOT))

PC1/PC2 erklaerte Varianz: [0.24573587 0.09129079]


Gespeichert: reports/figures/pca_cluster_final.png


## 7) Profilierungsvariablen (C_profile_only)

In [17]:
profile_rows = []
for var in C_profile_only:
    tmp = (
        df.groupby('cluster_final')[var]
        .value_counts(dropna=False, normalize=True)
        .mul(100).round(2)
        .rename('pct').reset_index()
        .rename(columns={var: 'response'})
    )
    tmp.insert(0, 'profile_variable', var)
    profile_rows.append(tmp[['profile_variable', 'cluster_final', 'response', 'pct']])

profile_table = pd.concat(profile_rows, ignore_index=True)
profile_table['response'] = profile_table['response'].astype(str)
profile_table = profile_table.rename(columns={'cluster_final': 'cluster'})
profile_table.to_csv(REPORTS_TABLES / '10_profile_variables_k2_distribution.csv', index=False)
print("Gespeichert:", (REPORTS_TABLES / "10_profile_variables_k2_distribution.csv").relative_to(OUTPUT_ROOT))
profile_table.head(15)

Gespeichert: reports/tables/10_profile_variables_k2_distribution.csv


,profile_variable,cluster,response,pct
0,Are you self-employed?,0,0,78.41
1,Are you self-employed?,0,1,21.59
2,Are you self-employed?,1,0,82.20
3,Are you self-employed?,1,1,17.80
4,Have you observed or experienced an unsupporti...,0,No,33.21
5,Have you observed or experienced an unsupporti...,0,Maybe/Not sure,24.44
6,Have you observed or experienced an unsupporti...,0,"Yes, I observed",20.05
7,Have you observed or experienced an unsupporti...,0,"Yes, I experienced",18.27
8,Have you observed or experienced an unsupporti...,0,nan,4.03
9,Have you observed or experienced an unsupporti...,1,No,48.64


## 8) HR-Maßnahmen und Abschluss

In [18]:
# Clusterbezeichnung datenbasiert ableiten (kein hartcodiertes Label)
burden_score = (
    cluster_profile_k2['mh_past_ord']
    + cluster_profile_k2['mh_current_ord']
    + cluster_profile_k2['mh_diagnosed_bin']
    + cluster_profile_k2['mh_treatment_bin']
    + cluster_profile_k2['interfere_untreated_ord']
)
high_burden_cluster = int(burden_score.idxmax())
low_burden_cluster  = int(burden_score.idxmin())

print(f'Cluster mit hoher Belastung: {high_burden_cluster}')
print(f'Cluster mit geringer Belastung: {low_burden_cluster}')

hr_rows = [
    (high_burden_cluster, 'Vertrauliche psychosoziale Erstberatung (EAP)'),
    (high_burden_cluster, 'Klare Kommunikation zu Vertraulichkeit und Nicht-Diskriminierung'),
    (high_burden_cluster, 'Fuehrungskraefte-Training: mentale Gesundheit adressieren'),
    (high_burden_cluster, 'Niedrigschwellige Wiedereingliederungsangebote'),
    (low_burden_cluster,  'Praevention: Resilienztrainings, Stressmanagement'),
    (low_burden_cluster,  'Arbeitsorganisation pruefen: Remote-Optionen, Workload'),
    (low_burden_cluster,  'Regelmaessige Pulsbefragung zur Fruehwarnung'),
]
hr_actions = pd.DataFrame(hr_rows, columns=['cluster', 'empfohlene_massnahme'])
hr_actions.to_csv(REPORTS_TABLES / '11_hr_actions_by_cluster.csv', index=False)
print('\nHR-Massnahmen je Cluster:')
print(hr_actions.to_string(index=False))

Cluster mit hoher Belastung: 0
Cluster mit geringer Belastung: 1

HR-Massnahmen je Cluster:
 cluster                                             empfohlene_massnahme
       0                    Vertrauliche psychosoziale Erstberatung (EAP)
       0 Klare Kommunikation zu Vertraulichkeit und Nicht-Diskriminierung
       0        Fuehrungskraefte-Training: mentale Gesundheit adressieren
       0                   Niedrigschwellige Wiedereingliederungsangebote
       1                Praevention: Resilienztrainings, Stressmanagement
       1           Arbeitsorganisation pruefen: Remote-Optionen, Workload
       1                     Regelmaessige Pulsbefragung zur Fruehwarnung


In [19]:
# Checkpoints speichern
objects_checkpoint = {
    'decision_df':              decision_df,
    'cluster_core_v5_primary':  cluster_core_v5_primary,
    'cluster_core_v5_sensitivity': cluster_core_v5_sensitivity,
    'method_eval_df':           method_eval_df,
    'cluster_profile_k2':       cluster_profile_k2,
    'cluster_profile_k3':       cluster_profile_k3,
    'cluster_diff_final':       cluster_diff_final,
    'medoid_indices':           medoid_indices,
    'stability_df':             stability_df,
    'scaler_v5':                scaler_v5,
    'pca':                      pca,
}
joblib.dump(df,                 CHECKPOINT_DIR / 'df_checkpoint_latest.pkl')
joblib.dump(objects_checkpoint, CHECKPOINT_DIR / 'objects_checkpoint_latest.pkl')
print('Checkpoints gespeichert.')

Checkpoints gespeichert.


## Zusammenfassung

Das Notebook bildet die Aufgabenstellung vollstaendig ab und dokumentiert einen fundierten
Methodenvergleich:

- **EDA**: strukturelle Missing-Value-Logik erkannt und fuer die Modellentscheidung genutzt
- **Feature Engineering**: Alter (skaliert), Geschlecht (robuste Wort-Boundary-Gruppen),
  Rolle (Multi-Hot), 13 Kernvariablen (ordinal/binaer kodiert), NA-Flags fuer strukturelle Luecken
- **Clustering-Diagnostik**: drei Verfahren (k-Means euklidisch, PAM/Gower, HAC/Gower-average)
  jeweils ueber k=2..8 mit vier Kennzahlen – konsistentes Optimum bei k=2
- **Finale Wahl**: PAM / FasterPAM auf Gower-Distanz mit k=2. Silhouette 0,36 (gegenueber
  0,20 im k-Means-Baseline), stabil ueber fuenf Seeds, mit realen Befragten als Medoids
- **Profilierung**: klar trennende Mental-Health-Dimensionen – belastungsnahe vs. gering
  betroffene Gruppe – plus sekundaerer Gender- und Rollenbefund
- **HR-Massnahmen**: datenbasierte Ableitung getrennter Handlungsschienen je Cluster
- **Reproduzierbarkeit**: Seeds fixiert, Checkpoints gespeichert, alle Exporte referenziert

**Code-Link:** <https://github.com/AndreiWilke/DLBDSMLUSL01-unsupervised-learning-fallstudie>